In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np

@dataclass
class Experience:
    state:      np.ndarray
    action:     int
    reward:     float
    next_state: np.ndarray
    done:       bool

In [3]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, experience: Experience):
        self.buffer.append(experience)
    
    def sample(self, batch_size: int) -> list[Experience]:
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

In [4]:
@dataclass(frozen=True)
class DuelingDQN_Params_Config:
    state_size:  int
    action_size: int
    hidden_size: int
    
@dataclass(frozen=True)
class DQN_Params_Config:
    lr:         float
    gamma:      float
    epsilon:    float
    eps_min:    float
    eps_decay:  float
    batch_size: int
    target_update_freq: int

@dataclass(frozen=True)
class DQN_Planner_Config:
    root_dir:   Path
    model_dir:  Path
    
    duel_dqn_params:    DuelingDQN_Params_Config
    dqn_params:         DQN_Params_Config

In [5]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_dqn_planner_config(self) -> DQN_Planner_Config:
        config = self.config.dqn_planner_config
        params = self.params.dqn_planner_params
        create_directories([config.root_dir])
        
        return DQN_Planner_Config(
            root_dir  = Path(config.root_dir),
            model_dir = Path(config.root_dir) / "dqn_model.pth",
            
            duel_dqn_params = DuelingDQN_Params_Config(
                state_size  = params.state_size,
                action_size = params.action_size,
                hidden_size = params.hidden_size
            ),
            
            dqn_params = DQN_Params_Config(
                lr         = params.lr,
                gamma      = params.gamma,
                epsilon    = params.epsilon,
                eps_min    = params.epsilon_min,
                eps_decay  = params.epsilon_decay,
                batch_size = params.batch_size,
                target_update_freq = params.target_update_freq
            )
        )

In [6]:
import torch
import torch.nn as nn
import numpy as np

from core.logging import logger

# Q(s,a) = V(s) + A(s,a) - mean(A(s,a))
class DuelingDQN(nn.Module):
    def __init__(self, state_size: int, action_size: int, hidden_size: int = 128):
        super().__init__()

        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        
        # Value stream V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Advantage stream A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
        
    def forward(self, x):
        shared = self.shared(x)
        value  = self.value_stream(shared)
        advantage = self.advantage_stream(shared)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

In [7]:
ACTIONS = {
    0: "do_nothing",
    1: "trigger_retraining",
    2: "switch_to_lighter_model",
    3: "scale_up_service",
    4: "restart_service",
}

In [ ]:
import torch
import torch.nn.functional as F

class DQNPlanner:
    def __init__(self, config: DQN_Planner_Config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.epsilon = self.config.dqn_params.epsilon
        
        self.online_net = self._get_network().to(self.device)
        self.target_net = self._get_network().to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()
        
        self.optimizer  = torch.optim.Adam(self.online_net.parameters(), lr=self.config.dqn_params.lr)
        self.buffer     = ReplayBuffer()
        self._ready     = False
        self.step_count = 0
    
    def _get_network(self):
        return DuelingDQN(
            state_size  = self.config.duel_dqn_params.state_size,
            action_size = self.config.duel_dqn_params.action_size,
            hidden_size = self.config.duel_dqn_params.hidden_size
        )
        
    # --------------------------- Inference ------------------------------------
    def plan(self, state: dict) -> str:
        state_tensor = self._dict_to_tensor(state)
        
        if random.random() < self.epsilon:
            action_idx = random.randint(0, self.config.duel_dqn_params.action_size - 1)
        else:
            with torch.no_grad():
                q_val       = self.online_net(state_tensor)
                action_idx  = q_val.argmax().item()
        
        action = ACTIONS[action_idx] 
        logger.info(f"DQN action: {action} (epsilon={self.epsilon:.3f})")
        return action
    
    def _dict_to_tensor(self, state: dict) -> torch.Tensor:
        values = (
            state.get("current_cpu",         [0.0]) +
            state.get("current_ram",         [0.0]) +
            state.get("current_latency",     [0.0]) +
            state.get("current_drift",       [0.0]) +
            state.get("predicted_cpu",       [0.0]*4) +
            state.get("predicted_ram",       [0.0]*4) +
            state.get("predicted_latency",   [0.0]*4) +
            state.get("predicted_drift",     [0.0]*4)
        )
        return torch.tensor(values, dtype=torch.float32).unsqueeze(0).to(self.device)
    # ------------------------------------------------------------------------------------
    
    # ----------------------- Training DQN (simulation) -------------------------------
    def train(self, n_episodes: int = 1000):
        from dqn.components.simulation import SystemSimulation
        self.simulation = SystemSimulation()
        
        for episode in range(n_episodes):
            total_reward = self._learn_and_reward()
            
            # Decay epsilon
            self.epsilon = max(
                self.config.dqn_params.eps_min, 
                self.epsilon * self.config.dqn_params.eps_decay
            )
            
            if episode % 100 == 0:
                logger.info(f"Episode {episode}/{n_episodes} — Reward: {total_reward:.2f} — Epsilon: {self.epsilon:.3f}")

        self._save()
        self._ready = True
    
    def _learn_and_reward(self):
        state = self.simulation.reset()
        total_reward = 0
            
        for _ in range(200):
            action_idx = self._select_action(state)
            next_state, reward, done = self.simulation.step(action_idx)
                
            self.buffer.push(Experience(state, action_idx, reward, next_state, done))
            self._learn()
                
            state = next_state
            total_reward += reward
            self.step_count += 1
                
            if self.step_count % self.config.dqn_params.target_update_freq == 0:
                self.target_net.load_state_dict(self.online_net.state_dict())
                
            if done:
                break
        return total_reward
    
    
    def _select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.config.duel_dqn_params.action_size - 1)
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self.online_net(state_tensor).argmax().item()
    
    
    def _learn(self):
        if len(self.buffer) < self.config.dqn_params.batch_size:
            return
        
        batch       = self.buffer.sample(self.config.dqn_params.batch_size)
        states      = torch.tensor(np.array([e.state  for e in batch]), dtype=torch.float32).to(self.device)
        actions     = torch.tensor(np.array([e.action for e in batch]), dtype=torch.long).to(self.device)
        rewards     = torch.tensor(np.array([e.reward for e in batch]), dtype=torch.float32).to(self.device)
        dones       = torch.tensor(np.array([e.done   for e in batch]), dtype=torch.float32).to(self.device)
        next_states = torch.tensor(np.array([e.next_state for e in batch]), dtype=torch.float32).to(self.device)
        
        # Double DQN: online net chooses action
        with torch.no_grad():
            next_actions = self.online_net(next_states).argmax(dim=1)
            next_q       = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            target_q     = rewards + self.config.dqn_params.gamma * next_q * (1 - dones)
        
        current_q = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        loss      = F.smooth_l1_loss(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), 10)
        self.optimizer.step()
    
    def _save(self):
        torch.save({
            "online_net":  self.online_net.state_dict(),
            "target_net":  self.target_net.state_dict(),
            "epsilon":     self.epsilon,
            "step_count":  self.step_count,
        }, self.config.model_dir)
        
        logger.info(f"DQN saved -> {self.config.model_dir}")

In [13]:
try:
    config = ConfigManager()
    planner = DQNPlanner(config.get_dqn_planner_config())
    planner.train()
except Exception as e:
    logger.error(f"Error: {e}")
    raise

[2026-04-19 20:23:19,079: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-19 20:23:19,086: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-19 20:23:19,088: INFO: __init__: created directory at: artifacts]
[2026-04-19 20:23:19,089: INFO: __init__: created directory at: artifacts/dqn_planner]
[2026-04-19 20:23:19,968: INFO: 1299283435: Episode 0/1000 — Reward: 34.00 — Epsilon: 0.995]
[2026-04-19 20:24:47,898: INFO: 1299283435: Episode 100/1000 — Reward: 105.00 — Epsilon: 0.603]
[2026-04-19 20:26:00,783: INFO: 1299283435: Episode 200/1000 — Reward: -6.00 — Epsilon: 0.365]
[2026-04-19 20:27:32,477: INFO: 1299283435: Episode 300/1000 — Reward: 116.00 — Epsilon: 0.221]
[2026-04-19 20:29:38,886: INFO: 1299283435: Episode 400/1000 — Reward: 182.00 — Epsilon: 0.134]
[2026-04-19 20:31:40,345: INFO: 1299283435: Episode 500/1000 — Reward: 188.00 — Epsilon: 0.081]
[2026-04-19 20:33:43,615: INFO: 1299283435: Episode 600/1000 — Reward: 186.00 — Epsilon